# FinAgent-RedRange — v0.1 demo

A reproducible walkthrough for reviewers: each attack **lands with controls off** and is
**blocked with controls on**, with every finding mapped to OWASP / MITRE ATLAS / NIST.

> Defensive research against a self-contained **mock** banking agent. All accounts and data
> are synthetic. The only target is the bundled agent in `src/finagent_redrange/target/`.
> See `SECURITY.md` for the responsible-disclosure posture.

Runs fully offline on a deterministic `EchoClient` — no API key required.

## 0. Setup — put the package on the path and build the agent factory

In [ ]:
import sys
from pathlib import Path

# Locate the repo root (works whether the kernel starts in notebooks/ or the repo root)
# and make the package importable without a prior `pip install -e .`.
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'finagent_redrange').is_dir()
)
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from finagent_redrange.llm.client import EchoClient  # swap for AnthropicClient for live results
from finagent_redrange.target.agent import BankingAgent, KnowledgeStore
from finagent_redrange.target.guardrails import Guardrails
from finagent_redrange.target.tools import Session, ToolRegistry

KNOWLEDGE = SRC / 'finagent_redrange' / 'target' / 'knowledge'

def build(controls_on: bool) -> BankingAgent:
    """A fresh agent per run so poisoned state never leaks between passes."""
    return BankingAgent(
        llm=EchoClient(),
        knowledge=KnowledgeStore.from_dir(KNOWLEDGE),
        guardrails=Guardrails(enabled=controls_on),
        tools=ToolRegistry(session=Session(user='alice', account_id='ACC-1001')),
    )

print('Agent factory ready. Session: alice / ACC-1001. LLM: EchoClient (offline).')

## 1. Indirect prompt injection → cross-account PII disclosure

An attacker who can influence a **retrieved** source (an ingested web page, a shared note)
plants an instruction inside it. The customer asks an innocent question; the agent retrieves
the poisoned chunk and follows the embedded instruction — leaking another customer's account.

In [ ]:
from finagent_redrange.scenarios.indirect_prompt_injection import (
    IndirectPromptInjectionScenario, POISONED_DOC, TARGET_ACCOUNT,
)

print('Injected policy update — references', TARGET_ACCOUNT, "(NOT alice's ACC-1001):")
print()
print(POISONED_DOC)

### Controls OFF — the POC lands

In [ ]:
from finagent_redrange.attacker.engine import run_campaign
from finagent_redrange.types import Role

def show(finding):
    retrieved = [t.content for t in finding.transcript.turns if t.role is Role.TOOL]
    print('RETRIEVED CONTEXT (what reached the model):')
    for r in retrieved:
        print('  -', ' '.join(r.split())[:120])
    print()
    print('ASSISTANT ANSWER:')
    print(' ', ' '.join(finding.transcript.assistant_text().split())[:200])
    print()
    print('ORACLE  ->', 'EXPLOITED' if finding.succeeded else 'blocked')

ipi_off = run_campaign(IndirectPromptInjectionScenario(), build(controls_on=False))
show(ipi_off)
print('OWASP   ->', ipi_off.frameworks.owasp_llm + ipi_off.frameworks.owasp_agentic)
print('ATLAS   ->', ipi_off.frameworks.mitre_atlas)
print('AIRQ    ->', ipi_off.airq.composite, ipi_off.airq.band.value)

### Controls ON — the mitigation blocks it

With controls enabled, the **retrieval-provenance check** drops the untrusted chunk and the
**output PII filter** redacts any account that isn't the session's — so the oracle flips.

In [ ]:
ipi_on = run_campaign(IndirectPromptInjectionScenario(), build(controls_on=True))
show(ipi_on)
print('Validating control ->', ipi_on.validating_control)

## 2. Data poisoning → fabricated transfer policy

Distinct from injection: here the attacker corrupts the **knowledge** the agent relies on, so
the agent confidently states a fabricated policy as fact — no malicious instruction needed.
The control is a **source allowlist + corpus integrity hash**.

In [ ]:
from finagent_redrange.scenarios.data_poisoning import DataPoisoningScenario, FALSE_POLICY

print('Poisoned policy (from an untrusted mirror, not the signed corpus):')
print()
print(FALSE_POLICY)
print()
dp_off = run_campaign(DataPoisoningScenario(), build(controls_on=False))
dp_on  = run_campaign(DataPoisoningScenario(), build(controls_on=True))
print('controls OFF ->', 'EXPLOITED' if dp_off.succeeded else 'blocked')
print('controls ON  ->', 'EXPLOITED' if dp_on.succeeded else 'blocked')
print('Validating control ->', dp_on.validating_control)

## 3. The scorecard — before / after, mapped to frameworks

The headline artifact: every scenario `exploited` with controls off and `blocked` with
controls on, each row carrying its OWASP / ATLAS IDs and AIRQ risk band.

In [ ]:
from IPython.display import Markdown, display
from finagent_redrange.scoring import scorecard

display(Markdown(scorecard.render_markdown([ipi_off, dp_off], [ipi_on, dp_on])))